In [ ]:
!mkdir -p ~/.kaggle && echo "KGAT_e01a71dc0cf0a380fe96a273e8b51c64" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
!pip install -q --upgrade kaggle ultralytics scikit-learn
!kaggle competitions list -s airbus-ship-detection

!mkdir -p /content/sanchay_32k
!cd /content/sanchay_32k && kaggle competitions download -c airbus-ship-detection -f train_ship_segmentations_v2.csv
!cd /content/sanchay_32k && unzip -q -o train_ship_segmentations_v2.csv.zip && rm -f train_ship_segmentations_v2.csv.zip
!cd /content/sanchay_32k && kaggle competitions download -c airbus-ship-detection
!ls -lh /content/sanchay_32k

ref                                                        deadline             category      reward  teamCount  userHasEntered  userRank  
---------------------------------------------------------  -------------------  --------  ----------  ---------  --------------  --------  
https://www.kaggle.com/competitions/airbus-ship-detection  2018-11-14 23:59:00  Featured  60,000 Usd        878            True         0  
100% 17.9M/17.9M [00:00<00:00, 53.6MB/s]

100% 28.6G/28.6G [05:09<00:00, 99.1MB/s]

total 29G
-rw-r--r-- 1 root root 29G Dec 16  2019 airbus-ship-detection.zip
-rw-r--r-- 1 root root 44M Dec 15  2019 train_ship_segmentations_v2.csv


In [ ]:
import glob, os, zipfile, numpy as np, pandas as pd

WORK_DIR = "/content/sanchay_32k"
RAW_DIR = os.path.join(WORK_DIR, "raw_imgs")
CSV_PATH = os.path.join(WORK_DIR, "train_ship_segmentations_v2.csv")
N_SHIP = 30000
N_NOSHIP = 2000
SEED = 42

df = pd.read_csv(CSV_PATH)
has_mask = df["EncodedPixels"].notna() & (df["EncodedPixels"].str.strip() != "")

ship_ids = sorted(df.loc[has_mask, "ImageId"].unique().tolist())
all_ids = sorted(df["ImageId"].unique().tolist())
ship_set = set(ship_ids)
noship_ids = [img_id for img_id in all_ids if img_id not in ship_set]

rng = np.random.default_rng(SEED)
chosen_ship = set(rng.choice(ship_ids, size=min(N_SHIP, len(ship_ids)), replace=False).tolist())
chosen_noship = set(rng.choice(noship_ids, size=min(N_NOSHIP, len(noship_ids)), replace=False).tolist())
selected_ids = chosen_ship | chosen_noship

# Save lookup for Step 3
pd.Series(sorted(chosen_ship), name="ImageId").to_csv(os.path.join(WORK_DIR, "ship_ids.csv"), index=False)
pd.Series(sorted(chosen_noship), name="ImageId").to_csv(os.path.join(WORK_DIR, "noship_ids.csv"), index=False)
print(f"Target selection -> Ship: {len(chosen_ship)}, No-ship: {len(chosen_noship)}, Total: {len(selected_ids)}")

zips = [z for z in glob.glob(os.path.join(WORK_DIR, "*.zip")) if "train_ship_segmentations_v2" not in z]
assert zips, "No competition zip found in /content/sanchay_32k"
comp_zip = max(zips, key=os.path.getsize)

os.makedirs(RAW_DIR, exist_ok=True)
extracted = 0

with zipfile.ZipFile(comp_zip, "r") as zf:
    nested = [n for n in zf.namelist() if n.endswith("train_v2.zip")]
    if nested:
        zf.extract(nested[0], "/tmp")
        with zipfile.ZipFile(f"/tmp/{nested[0]}", "r") as nzf:
            for member in nzf.namelist():
                fname = os.path.basename(member)
                if fname in selected_ids:
                    with open(os.path.join(RAW_DIR, fname), "wb") as out_f, nzf.open(member) as in_f:
                        out_f.write(in_f.read())
                    extracted += 1
        os.remove(f"/tmp/{nested[0]}")
    else:
        for member in zf.namelist():
            fname = os.path.basename(member)
            if fname in selected_ids:
                with open(os.path.join(RAW_DIR, fname), "wb") as out_f, zf.open(member) as in_f:
                    out_f.write(in_f.read())
                extracted += 1

os.remove(comp_zip)
print("Extracted images:", extracted)

Target selection -> Ship: 30000, No-ship: 2000, Total: 32000
Extracted images: 32000


In [ ]:
import os, shutil, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

WORK_DIR = "/content/sanchay_32k"
RAW_DIR = os.path.join(WORK_DIR, "raw_imgs")
YOLO_DIR = os.path.join(WORK_DIR, "yolo_dataset")
CSV_PATH = os.path.join(WORK_DIR, "train_ship_segmentations_v2.csv")
IMG_W, IMG_H = 768, 768
SEED = 42


def rle_to_bbox(mask_rle: str, shape=(IMG_H, IMG_W)):
    s = mask_rle.strip().split()
    starts = np.asarray(s[0::2], dtype=np.int64) - 1
    lengths = np.asarray(s[1::2], dtype=np.int64)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    mask = img.reshape((shape[1], shape[0])).T
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min, x_max = xs.min(), xs.max() + 1
    y_min, y_max = ys.min(), ys.max() + 1

    xc = np.clip(((x_min + x_max) / 2.0) / shape[1], 0.0, 1.0)
    yc = np.clip(((y_min + y_max) / 2.0) / shape[0], 0.0, 1.0)
    w = np.clip((x_max - x_min) / shape[1], 0.0, 1.0)
    h = np.clip((y_max - y_min) / shape[0], 0.0, 1.0)
    return (xc, yc, w, h) if (w > 0 and h > 0) else None


df = pd.read_csv(CSV_PATH)
ship_df = df[df["EncodedPixels"].notna() & (df["EncodedPixels"].str.strip() != "")]
grouped = {img_id: grp["EncodedPixels"].tolist() for img_id, grp in ship_df.groupby("ImageId")}

available_files = sorted(os.listdir(RAW_DIR))
# Stratify split by whether the image has ships
is_ship_flags = [1 if fn in grouped else 0 for fn in available_files]
train_files, val_files = train_test_split(
    available_files, test_size=0.2, random_state=SEED, stratify=is_ship_flags
)

for split, split_files in [("train", train_files), ("val", val_files)]:
    img_out = os.path.join(YOLO_DIR, "images", split)
    lbl_out = os.path.join(YOLO_DIR, "labels", split)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    ship_cnt, noship_cnt = 0, 0
    for fn in split_files:
        lbl_path = os.path.join(lbl_out, os.path.splitext(fn)[0] + ".txt")
        if fn in grouped:
            boxes = [b for b in (rle_to_bbox(r) for r in grouped[fn]) if b is not None]
            if not boxes:
                continue
            shutil.move(os.path.join(RAW_DIR, fn), os.path.join(img_out, fn))
            with open(lbl_path, "w") as f:
                for xc, yc, w, h in boxes:
                    f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
            ship_cnt += 1
        else:
            shutil.move(os.path.join(RAW_DIR, fn), os.path.join(img_out, fn))
            open(lbl_path, "w").close()
            noship_cnt += 1
    print(f"{split}: {ship_cnt} ship + {noship_cnt} no-ship = {ship_cnt + noship_cnt} ready")

shutil.rmtree(RAW_DIR, ignore_errors=True)

yaml_path = os.path.join(YOLO_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\n\nnc: 1\nnames: ['ship']\n")
print("Saved:", yaml_path)

In [ ]:
import os, shutil, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

WORK_DIR = "/content/sanchay_32k"
RAW_DIR = os.path.join(WORK_DIR, "raw_imgs")
YOLO_DIR = os.path.join(WORK_DIR, "yolo_dataset")
CSV_PATH = os.path.join(WORK_DIR, "train_ship_segmentations_v2.csv")
IMG_W, IMG_H = 768, 768
SEED = 42


def rle_to_bbox(mask_rle: str, shape=(IMG_H, IMG_W)):
    s = mask_rle.strip().split()
    starts = np.asarray(s[0::2], dtype=np.int64) - 1
    lengths = np.asarray(s[1::2], dtype=np.int64)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    mask = img.reshape((shape[1], shape[0])).T
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min, x_max = xs.min(), xs.max() + 1
    y_min, y_max = ys.min(), ys.max() + 1

    xc = np.clip(((x_min + x_max) / 2.0) / shape[1], 0.0, 1.0)
    yc = np.clip(((y_min + y_max) / 2.0) / shape[0], 0.0, 1.0)
    w = np.clip((x_max - x_min) / shape[1], 0.0, 1.0)
    h = np.clip((y_max - y_min) / shape[0], 0.0, 1.0)
    return (xc, yc, w, h) if (w > 0 and h > 0) else None


df = pd.read_csv(CSV_PATH)
ship_df = df[df["EncodedPixels"].notna() & (df["EncodedPixels"].str.strip() != "")]
grouped = {img_id: grp["EncodedPixels"].tolist() for img_id, grp in ship_df.groupby("ImageId")}

available_files = sorted(os.listdir(RAW_DIR))
is_ship_flags = [1 if fn in grouped else 0 for fn in available_files]
train_files, val_files = train_test_split(
    available_files, test_size=0.2, random_state=SEED, stratify=is_ship_flags
)

for split, split_files in [("train", train_files), ("val", val_files)]:
    img_out = os.path.join(YOLO_DIR, "images", split)
    lbl_out = os.path.join(YOLO_DIR, "labels", split)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    ship_cnt, noship_cnt = 0, 0
    for fn in split_files:
        lbl_path = os.path.join(lbl_out, os.path.splitext(fn)[0] + ".txt")
        if fn in grouped:
            boxes = [b for b in (rle_to_bbox(r) for r in grouped[fn]) if b is not None]
            if not boxes:
                continue
            shutil.move(os.path.join(RAW_DIR, fn), os.path.join(img_out, fn))
            with open(lbl_path, "w") as f:
                for xc, yc, w, h in boxes:
                    f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
            ship_cnt += 1
        else:
            shutil.move(os.path.join(RAW_DIR, fn), os.path.join(img_out, fn))
            open(lbl_path, "w").close()
            noship_cnt += 1
    print(f"{split}: {ship_cnt} ship + {noship_cnt} no-ship = {ship_cnt + noship_cnt} ready")

shutil.rmtree(RAW_DIR, ignore_errors=True)

yaml_path = os.path.join(YOLO_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\n\nnc: 1\nnames: ['ship']\n")
print("Saved:", yaml_path)

train: 24000 ship + 1600 no-ship = 25600 ready
val: 6000 ship + 400 no-ship = 6400 ready
Saved: /content/sanchay_32k/yolo_dataset/data.yaml


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


In [ ]:
from ultralytics import YOLO
from google.colab import files

model = YOLO("yolov8s.pt")
model.train(
    data="/content/sanchay_32k/yolo_dataset/data.yaml",
    epochs=15,
    imgsz=768,
    batch=16,
    project="/content/sanchay_32k/runs",
    name="ship_30k_plus_2k",
    device=0,
)

files.download("/content/sanchay_32k/runs/ship_30k_plus_2k/weights/best.pt")
files.download("/content/sanchay_32k/runs/ship_30k_plus_2k/results.csv")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sanchay_32k/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
WEIGHTS_PATH = "/content/sanchay_32k/runs/ship_30k_plus_2k/weights/best.pt"
IMG_DIR = "/content/sanchay_32k/yolo_dataset/images/val"

In [ ]:
#testing
import os
from google.colab import files

os.makedirs("data/port_images", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join("data/port_images", name), "wb") as f:
        f.write(data)

Saving GE_1.jpg to GE_1.jpg


In [ ]:
import os, numpy as np, pandas as pd, torch
from ultralytics import YOLO

# Use local best.pt if present, otherwise fall back to the Colab training run path
WEIGHTS_PATH = "best.pt" if os.path.exists("best.pt") else "/content/sanchay_32k/runs/ship_30k_plus_2k/weights/best.pt"
IMG_PATH = "GE_1.jpg"

assert os.path.exists(WEIGHTS_PATH), f"Weights not found: {WEIGHTS_PATH}"
assert os.path.exists(IMG_PATH), f"Image not found: {IMG_PATH}"

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(WEIGHTS_PATH)

# Try conf=0.35 first; if high-res Google Earth ships look smaller/different, you can lower conf (e.g., 0.25)
results = model.predict(source=IMG_PATH, conf=0.35, iou=0.5, imgsz=768, device=device, verbose=False)
res = results[0]
res.save(filename="pred_GE_1.jpg")

boxes = res.boxes
n_ships = len(boxes) if boxes is not None else 0

if n_ships > 0:
    wh = boxes.xywhn.cpu().numpy()
    areas = wh[:, 2] * wh[:, 3]
    confs = boxes.conf.cpu().numpy()
    mean_area = float(np.mean(areas))
    total_area = float(np.sum(areas))
    mean_conf = float(np.mean(confs))
else:
    mean_area = 0.0
    total_area = 0.0
    mean_conf = 0.0

det_df = pd.DataFrame([{
    "image_id": os.path.basename(IMG_PATH),
    "ship_count": n_ships,
    "mean_box_area": mean_area,
    "total_box_area": total_area,
    "mean_conf": mean_conf,
}])
det_df.to_csv("vision_detections.csv", index=False)
print(f"Processed on device={device} using {WEIGHTS_PATH}")
print(det_df)
print("Saved annotated image to pred_GE_1.jpg")

Processed on device=0 using best.pt
   image_id  ship_count  mean_box_area  total_box_area  mean_conf
0  GE_1.jpg           2       0.003763        0.007527    0.61532
Saved annotated image to pred_GE_1.jpg


ML Model and features

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

df = pd.read_csv("sanchay_ml_features_v3.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "port_name"]).reset_index(drop=True)

# Strict chronological split matching v3 experiments
train_df = df[df["date"] <= "2025-05-31"].copy()
test_df = df[df["date"] >= "2025-07-01"].copy()

exclude_cols = {
    "record_id", "date", "charter_action", "recommended_alt_vessel_class",
    "true_vessel_count", "true_congestion_index",
    "freight_rate_next_7d_usd", "freight_rate_next_30d_usd", "freight_rate_next_90d_usd",
    "benchmark_next_7d_usd_mt", "benchmark_next_30d_usd_mt", "benchmark_next_90d_usd_mt",
    "target_logret_7d", "target_logret_30d", "target_logret_90d",
    "expected_delay_days", "expected_demurrage_cost_usd",
    "alt_freight_rate_usd", "potential_savings_usd", "total_freight_cost_inr",
}

cv_cols = {
    "cv_obs_available", "cv_modality", "cv_ship_count", "cv_mean_conf",
    "days_since_cv_obs", "anchorage_vessel_count", "port_congestion_index",
    "anchorage_7d_rolling_mean", "cv_source", "imaging_source", "satellite_cloud_cover_pct",
} | {c for c in df.columns if c.startswith("anchorage_vessel_count_")}

all_features = [c for c in df.columns if c not in exclude_cols]
no_cv_features = [c for c in all_features if c not in cv_cols]


def prep_X(sub_df, cols):
    X = sub_df[cols].copy()
    for col in X.select_dtypes(include=["object", "category"]).columns:
        X[col] = X[col].astype("category")
    return X


def eval_models(target_col, feature_list, label):
    tr = train_df.dropna(subset=[target_col])
    te = test_df.dropna(subset=[target_col])
    X_tr, y_tr = prep_X(tr, feature_list), tr[target_col]
    X_te, y_te = prep_X(te, feature_list), te[target_col]

    models = {
        "HistGBM": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.04, random_state=42),
    }
    try:
        from lightgbm import LGBMRegressor
        models["LightGBM"] = LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, verbose=-1)
    except ImportError:
        pass
    try:
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(
            n_estimators=300, learning_rate=0.03, max_depth=6,
            enable_categorical=True, random_state=42, verbosity=0
        )
    except ImportError:
        pass

    print(f"\n=== Target: {target_col} ({label}) ===")
    for name, m in models.items():
        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)
        mae = mean_absolute_error(y_te, pred)
        rmse = root_mean_squared_error(y_te, pred)
        r2 = r2_score(y_te, pred)
        extra = ""
        if target_col == "target_logret_7d":
            pred_px = te["freight_benchmark_usd_mt"].values * np.exp(pred)
            true_px = te["benchmark_next_7d_usd_mt"].values
            extra = f" | 7d Bench MAE: {mean_absolute_error(true_px, pred_px):.3f} USD/MT"
        print(f"{name:10s} -> MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.3f}{extra}")


eval_models("target_logret_7d", no_cv_features, "Market + Calendar only")
eval_models("target_logret_7d", all_features, "With CV / Congestion")
eval_models("expected_delay_days", no_cv_features, "Market + Calendar only")
eval_models("expected_delay_days", all_features, "With CV / Congestion")


=== Target: target_logret_7d (Market + Calendar only) ===
HistGBM    -> MAE: 0.0390 | RMSE: 0.0526 | R2: -0.892 | 7d Bench MAE: 0.692 USD/MT
LightGBM   -> MAE: 0.0394 | RMSE: 0.0527 | R2: -0.898 | 7d Bench MAE: 0.702 USD/MT
XGBoost    -> MAE: 0.0358 | RMSE: 0.0489 | R2: -0.635 | 7d Bench MAE: 0.631 USD/MT

=== Target: target_logret_7d (With CV / Congestion) ===
HistGBM    -> MAE: 0.0377 | RMSE: 0.0509 | R2: -0.770 | 7d Bench MAE: 0.668 USD/MT
LightGBM   -> MAE: 0.0407 | RMSE: 0.0542 | R2: -1.009 | 7d Bench MAE: 0.725 USD/MT
XGBoost    -> MAE: 0.0332 | RMSE: 0.0455 | R2: -0.417 | 7d Bench MAE: 0.583 USD/MT

=== Target: expected_delay_days (Market + Calendar only) ===
HistGBM    -> MAE: 0.5924 | RMSE: 0.7784 | R2: 0.141
LightGBM   -> MAE: 0.5648 | RMSE: 0.7474 | R2: 0.208
XGBoost    -> MAE: 0.6275 | RMSE: 0.8152 | R2: 0.058

=== Target: expected_delay_days (With CV / Congestion) ===
HistGBM    -> MAE: 0.4583 | RMSE: 0.6161 | R2: 0.462
LightGBM   -> MAE: 0.4522 | RMSE: 0.6110 | R2: 0.471

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp

V3_XLSX = "Sanchay_Numeric_Dataset_v3.xlsx"
milp_params = pd.read_excel(V3_XLSX, sheet_name="MILP_Port_Vessel_Params")

# Filter to operationally & draft feasible port x vessel-class rows
feas = milp_params[
    (milp_params["class_allowed_at_port"] == 1) & (milp_params["draft_feasible"] == 1)
].copy().reset_index(drop=True)

# Example demand per port (MT) — replace with your actual shipment demand dict
port_demand_mt = {
    "Paradip": 120000.0,
    "Visakhapatnam": 90000.0,
    "Haldia": 30000.0,
    "Kandla": 50000.0,
    "Mundra": 140000.0,
    "Krishnapatnam": 80000.0,
    "Mormugao": 45000.0,
    "New Mangalore": 40000.0,
}

# Optional forecasted delay (days) per port from your forecaster; defaults to 1.0 day
pred_delay_by_port = {p: 1.0 for p in feas["port_name"].unique()}

# Decision variable x_i = number of vessels/voyages assigned to feasible row i (integer >= 0)
n_vars = len(feas)
demurrage = feas["demurrage_usd_day_median_last90d"].fillna(feas["demurrage_usd_day_median_all"]).values
delay_days = feas["port_name"].map(pred_delay_by_port).fillna(1.0).values
cargo_per_trip = feas["typical_cargo_mt"].clip(lower=1.0).values

# Objective per trip: port cost + expected delay/demurrage cost
c_obj = feas["port_cost_usd"].values + demurrage * delay_days

# Constraint 1: total moved cargo per port >= required demand (with reasonable upper cap)
ports_present = [p for p in port_demand_mt if p in set(feas["port_name"])]
A_rows, lb_rows, ub_rows = [], [], []

for p in ports_present:
    mask = (feas["port_name"] == p).values.astype(float)
    row_cargo = mask * cargo_per_trip
    A_rows.append(row_cargo)
    lb_rows.append(port_demand_mt[p])
    ub_rows.append(np.inf)

    # Constraint 2: berth capacity per port
    A_rows.append(mask)
    lb_rows.append(0.0)
    ub_rows.append(float(feas.loc[feas["port_name"] == p, "berth_capacity_vessels"].iloc[0]))

res = milp(
    c=c_obj,
    integrality=np.ones(n_vars),
    bounds=Bounds(lb=np.zeros(n_vars), ub=np.full(n_vars, 20.0)),
    constraints=LinearConstraint(np.array(A_rows), lb=np.array(lb_rows), ub=np.array(ub_rows)),
)

feas["assigned_trips"] = np.round(res.x).astype(int) if res.x is not None else 0
feas["allocated_cargo_mt"] = feas["assigned_trips"] * feas["typical_cargo_mt"]
plan = feas[feas["assigned_trips"] > 0][
    ["port_name", "vessel_class", "assigned_trips", "typical_cargo_mt", "allocated_cargo_mt", "port_max_draft_m", "vessel_draft_m"]
]
print("MILP status:", res.message, "| Total objective cost:", round(float(res.fun), 2))
print(plan.to_string(index=False))

MILP status: Optimization terminated successfully. (HiGHS Status 7: Optimal) | Total objective cost: 942000.0
    port_name vessel_class  assigned_trips  typical_cargo_mt  allocated_cargo_mt  port_max_draft_m  vessel_draft_m
      Paradip     Capesize               1            130500              130500              16.0           15.21
Visakhapatnam    Handysize               1             30250               30250              14.5            8.71
Visakhapatnam      Panamax               1             66500               66500              14.5           12.75
       Haldia    Handysize               1             30500               30500               9.0            8.74
       Kandla      Panamax               1             67000               67000              14.0           12.80
       Mundra    Handysize               1             30000               30000              17.0            8.67
       Mundra     Supramax               1             47500               47500     